# microlab — M0 gate run

Runs the M0 gate on Kaggle: **10M params on TinyStories to val loss < 1.6**, plus a
deliberate mid-run kill to prove the session-chaining harness recovers.

**Settings before you run:** Accelerator → **GPU T4 x2**, Internet → **on**.

Kaggle preempts every session at 12 hours. This notebook is written so that
**re-running it after a preemption continues the run** rather than restarting it —
that is the whole point of the checkpointing layer, and cell 5 tests it on purpose.


## 1. Install


In [ ]:
# Clone, or bring an existing clone exactly up to date.
#
# fetch + reset rather than `git pull`: the branch history has been rewritten
# once (authorship), and pull fails on divergence. reset --hard is
# unconditional and idempotent, so re-running this cell always lands on the
# remote tip. An earlier version put the update behind a `git clone ... ||`
# fallback, which silently did nothing on the second run.
import os
BRANCH = 'claude/implementation-task-list-w8ce0g'
if not os.path.exists('/kaggle/working/microlab'):
    !git clone -q --branch {BRANCH} https://github.com/achalshah20/microlab.git /kaggle/working/microlab
%cd /kaggle/working/microlab
!git fetch -q origin {BRANCH} && git reset --hard -q origin/{BRANCH}
!git log --oneline -1
!pip install -q -e ".[data,dev]"

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    # Report native bf16 (compute capability >= 8), not is_bf16_supported():
    # recent PyTorch returns True from that on a T4 because bf16 is emulated
    # in software. Emulated bf16 is far slower than the fp16 tensor cores.
    print(f'  gpu{i}: {p.name} sm_{p.major}{p.minor} {p.total_memory/1e9:.1f}GB '
          f'bf16_native={p.major >= 8} '
          f'(torch reports {torch.cuda.is_bf16_supported()})')
# Expect: Tesla T4, sm_75, bf16_native=False -> configs/m0.yaml uses fp16.


## 2. GPU correctness tests

These are the three tests CI cannot run. They check that fp16 loss-scaling skips a step
on non-finite gradients, and — importantly — that gradients are unscaled exactly *once*.
Double-unscaling trains fine and silently worse, so this is worth the minute it costs.


In [ ]:
!pytest -m gpu -q


## 3. Data

Tokenizer training is pure Python and takes a few minutes. It is written to
`/kaggle/working`, which does *not* survive a session — see cell 7 for persisting it.


In [ ]:
!python -m microlab.cli prepare \
    --source tinystories \
    --out-dir /kaggle/working/data/tinystories \
    --vocab-size 8192


## 4. Smoke run — 200 steps (~15 min)

**Do this before committing 6 hours.** Watch two numbers in the output:

- `loss_scale` — should settle high (2^15-ish) and stay put. A value that keeps
  halving means gradients are overflowing and the run will not survive.
- `skip_rate` — a handful of skipped steps early is healthy. A rate that *climbs*
  is divergence, visible here thousands of steps before the loss curve shows it.

Also check `mfu` — this is the first honest MFU number for the project, measured
against a denominator that is tested (`tests/test_shapes.py`), not assumed.


In [ ]:
!python scripts/kaggle_train.py \
    --config m0 \
    --data-dir /kaggle/working/data/tinystories \
    --runs-dir /kaggle/working/runs \
    train.run_id=m0_smoke train.max_steps=200 train.log_interval=10


In [ ]:
import json, pandas as pd
rows = [json.loads(l) for l in open('/kaggle/working/runs/m0_smoke/metrics.jsonl')]
df = pd.DataFrame([r for r in rows if 'loss' in r and 'val_loss' not in r])
print(df[['step','loss','lr','grad_norm','tokens_per_s','loss_scale','skip_rate','mfu']]
      .tail(10).to_string(index=False))
print('\nfinal skip rate:', df['skip_rate'].iloc[-1], '(want: small and flat)')
df.plot(x='step', y='loss', figsize=(9,3), title='smoke run loss');


## 5. Prove the chain recovers — kill it on purpose

The M0 gate says, verbatim: *kill the session mid-run and verify the chain recovers*.
`tests/test_session_chaining.py` already does this with SIGTERM in CI; this repeats it
on the real GPU with real checkpoints, which is the version that counts before M4
stakes ~50 sessions on it.

Start a run, kill it hard (SIGKILL — no clean shutdown, no final flush), then resume.


In [ ]:
import subprocess, time, signal, os, sys, json
from pathlib import Path

run = Path('/kaggle/working/runs/m0_killtest')
!rm -rf {run}

cmd = [sys.executable, '-m', 'microlab.cli', 'train', 'm0',
       'data.train_bin=/kaggle/working/data/tinystories/train.bin',
       'data.val_bin=/kaggle/working/data/tinystories/val.bin',
       'data.tokenizer_path=/kaggle/working/data/tinystories/tokenizer.json',
       'train.out_dir=/kaggle/working/runs', 'train.run_id=m0_killtest',
       'train.max_steps=300', 'train.checkpoint_every_steps=20',
       'train.log_interval=5', 'train.eval_interval=0']

p = subprocess.Popen(cmd)
while not list((run/'checkpoints').glob('*.meta.json')):
    time.sleep(2)
time.sleep(20)
p.send_signal(signal.SIGKILL)   # hardest possible kill: no handler runs
p.wait()
print('killed at:', sorted(x.name for x in (run/'checkpoints').glob('*.meta.json')))


In [ ]:
# Resume. A second process must continue from the checkpoint, not restart.
subprocess.run(cmd + ['train.max_steps=60'], check=True)

sessions = [json.loads(l) for l in open(run/'sessions.jsonl')]
begins = [s for s in sessions if s['event'] == 'begin']
steps = sorted({r['step'] for r in
                (json.loads(l) for l in open(run/'metrics.jsonl')) if 'loss' in r})

print('sessions:', len(begins))
print('resumed from step:', begins[1]['resumed_from_step'])
print('step history contiguous:', steps == list(range(1, max(steps)+1)))
assert len(begins) == 2 and begins[1]['resumed_from_step'] > 0, 'chain did NOT resume'
assert steps == list(range(1, max(steps)+1)), 'gap in metric history'
print('\nPASS — SIGKILL survived, run record continuous')


## 6. The gate run (~6 GPU-hours)

12000 steps x 65,536 tokens = ~790M tokens. Fits inside one 12-hour session with room,
so the gate itself does not depend on chaining — chaining is *tested* above, on a run
short enough that a failure costs an afternoon rather than a week.

If the session dies anyway, just re-run this cell: it resumes.


In [ ]:
!python scripts/kaggle_train.py \
    --config m0 \
    --data-dir /kaggle/working/data/tinystories \
    --runs-dir /kaggle/working/runs \
    train.run_id=m0


## 7. Gate check and artifacts


In [ ]:
import json
rows = [json.loads(l) for l in open('/kaggle/working/runs/m0/metrics.jsonl')]
vals = [r['val_loss'] for r in rows if 'val_loss' in r]
best = min(vals)
print(f'best val loss: {best:.4f}   gate: < 1.6   ->  {"PASS" if best < 1.6 else "FAIL"}')

sessions = [json.loads(l) for l in open('/kaggle/working/runs/m0/sessions.jsonl')]
print('sessions used:', sum(1 for s in sessions if s['event']=='begin'))


In [ ]:
!python -m microlab.cli sample m0 \
    data.tokenizer_path=/kaggle/working/data/tinystories/tokenizer.json \
    train.out_dir=/kaggle/working/runs \
    --prompt 'Once upon a time' --max-new-tokens 200


In [ ]:
# Full benchmark on the real hardware — the honest T4 numbers for the write-up.
!python -m microlab.cli bench m0 --steps 30 --warmup 5 --out /kaggle/working/bench_t4.json
!cat /kaggle/working/bench_t4.json


## 8. Keep the results

`/kaggle/working` is wiped when the session ends. The run record (metrics, session log,
samples) is small and is the evidence the gate was cleared — save it. Checkpoints are
large and belong on HF Hub (M1 makes that the default storage layer).


In [ ]:
!tar czf /kaggle/working/m0_run_record.tar.gz \
    --exclude='checkpoints' -C /kaggle/working/runs m0 m0_smoke m0_killtest
!ls -lh /kaggle/working/m0_run_record.tar.gz
# Download this from the notebook's Output tab, then commit it to the repo
# under runs/ so the gate result is reproducible from the record.
